### Preparation

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

### Q1. How many lesson pages

In [4]:
len(documents)

72

### Q2. Indexing and searching

In [6]:
from minsearch import Index

index = Index(text_fields = ['content'], keyword_fields = ['filename'])
index.fit(documents)

question = 'How does the agentic loop keep calling the model until it stops?'
index.search(question, num_results=1)

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

### Q3.RAG

In [18]:
import importlib
import rag

importlib.reload(rag)

RAGSearch = rag.RAGSearch

In [19]:
from openai import OpenAI

openai_client = OpenAI()
question = 'How does the agentic loop keep calling the model until it stops?'
rag_model = RAGSearch(index = index, llm_client = openai_client)
response, input_token = rag_model.rag(query = question)

In [20]:
response, input_token

('It keeps calling the model in a `while True` loop, and after each response it checks whether there were any `function_call` items. If there were, it runs the tool, appends the tool output to the message history, and calls the model again. It stops when a turn returns no function calls, i.e. the `has_function_calls` flag stays `False`.',
 7128)

### Q4.Chunking

In [22]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

### Q5.RAG with chunking

In [23]:
index = Index(text_fields = ['content'], keyword_fields = ['filename'])
index.fit(chunks)

In [25]:
importlib.reload(rag)
RAGSearch = rag.RAGSearch

question = 'How does the agentic loop keep calling the model until it stops?'
rag_model = RAGSearch(index = index, llm_client = openai_client)
response, input_token = rag_model.rag(query = question)

In [26]:
response, input_token

('The loop keeps calling the model inside a `while True` loop and checks whether any tool/function calls happened in that turn.\n\n- It sets `has_function_calls = False` before each model call.\n- It calls the model and processes `response.output`.\n- If it sees a `function_call`, it runs the tool, appends the result, and sets `has_function_calls = True`.\n- After the turn, if `has_function_calls == False`, it `break`s out of the loop.\n\nSo it stops when the model returns a response with no function calls.',
 2315)

### Q6. Turning it into an agent

In [40]:
from toyaikit.tools import Tools

In [66]:
def search(query: str):
    # index = Index(text_fields = ['content'], keyword_fields = ['filename'])
    # index.fit(chunks)
    return index.search(query, num_results=3)

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the index for relevant documents matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course content."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [67]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [68]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the index for relevant documents matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'Search query text to look up in the course content.'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [69]:
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback
from toyaikit.llm import OpenAIClient

In [70]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [71]:
instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [72]:
result = runner.loop(
    prompt = "How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback
)

-> Response received


-> Response received
